## Paso 1 — Descarga automática de datos (PRG, PO y TCO)

Este código automatiza la descarga diaria de archivos desde el portal **“Programas de Operación 2021”** del Coordinador Eléctrico, para un rango de fechas. Para cada día, realiza una búsqueda por fecha en la página, descarga los ZIP correspondientes y extrae los archivos relevantes, guardándolos en carpetas separadas.

### ¿Qué descarga y dónde lo guarda?

1. **PROGRAMAYYYYMMDD(.zip / -1 / -2)**  
   Desde este ZIP se extraen y almacenan:
   - `PRGYYMMDD.xlsx` → carpeta: `Programas/PRG/`
   - `POYYMMDD.xlsx`  → carpeta: `Programas/PO/`

2. **TCOYYYYMMDD.zip**  
   Desde este ZIP se extrae y almacena:
   - `TCOYYYYMMDD.xlsm` → carpeta: `TCO/`

3. **Carpeta temporal de trabajo**  
   - `_tmp_download/` se usa para descargar los ZIP y extraerlos antes de moverlos a su carpeta final.

### Flujo general (enumeración del proceso)

1. **Inicialización de carpetas**
   - Crea (si no existen) las carpetas de salida (`Programas/PRG`, `Programas/PO`, `TCO`) y la carpeta temporal (`_tmp_download`).

2. **Apertura del navegador y navegación al portal**
   - Inicia Selenium con Chrome y entra a la página de documentos.

3. **Iteración día a día**
   Para cada fecha dentro del rango:
   1. Se setean **fecha inicio** y **fecha término**
   2. Se presiona **“Realizar Búsqueda”** para actualizar la lista de documentos.
   3. Se intenta expandir **“Ver más”** para mostrar enlaces adicionales (si aplica).

4. **Descarga y extracción de PROGRAMA**
   1. Busca y descarga el ZIP `PROGRAMAYYYYMMDD.zip` (o versiones `-1`, `-2` si existen).
   2. Extrae el ZIP en la carpeta temporal.
   3. Mueve `PRGYYMMDD.xlsx` y `POYYMMDD.xlsx` a sus carpetas finales.

5. **Descarga y extracción de TCO**
   1. Busca y descarga `TCOYYYYMMDD.zip`.
   2. Extrae el ZIP en la carpeta temporal.
   3. Mueve `TCOYYYYMMDD.xlsm` a la carpeta final.

6. **Limpieza**
   - Elimina ZIPs y archivos temporales en `_tmp_download` para evitar confundir descargas de días distintos.

### Salida del paso
Al finalizar, quedan almacenados en disco los archivos PRG, PO y TCO correspondientes a cada día del rango, listos para el siguiente paso (carga/transformación en SQL).


## Descarga Generación Real

In [1]:
MESES = {
    1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"
}

# ---------------- CONFIG ----------------
SELECCIONAR_TODOS_ANIOS = False
ANIOS = [2026]  # ejemplo: [2021, 2022, 2023] o [] si no quieres seleccionar

SELECCIONAR_TODOS_MESES = False
MESES_SELECCION = [MESES[2]]  # ejemplo: [MESES[1], MESES[2]]  -> Enero, Febrero
# ----------------------------------------

from __future__ import annotations

import time
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

MASHUP_URL = "https://qap-prd.coordinador.cl/ext/extensions/mashup_Generacion_Real_Descargable/mashup_Generacion_Real_Descargable.html"

# Carpeta objetivo
DOWNLOAD_DIR = Path("/Users/valentinauretadelafuente/Desktop/Operacion Real/Datos Operación")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Nombre final (para evitar "(1)", "(2)")
FINAL_NAME = f"Generacion_real_{ANIOS[0]}_{MESES_SELECCION[0]}.csv"
# ----------------------------------------


def seleccionar_valores(driver, wait, valores=None, seleccionar_todos=False):
    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "[data-testid='listbox.item'] [role='row']")
    ))

    rows = driver.find_elements(By.CSS_SELECTOR, "[data-testid='listbox.item'] [role='row']")

    if seleccionar_todos:
        for row in rows:
            if row.get_attribute("aria-selected") != "true":
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", row)
                row.click()
                time.sleep(0.1)
        return

    if not valores:
        return

    for val in valores:
        val_text = str(val)
        row = wait.until(EC.element_to_be_clickable(
            (By.XPATH, f"//div[@role='row' and .//span[normalize-space()='{val_text}']]")
        ))
        if row.get_attribute("aria-selected") != "true":
            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", row)
            row.click()
            time.sleep(0.1)


def confirmar_seleccion(driver, wait, titulo):
    btn = wait.until(EC.element_to_be_clickable(
        (By.XPATH, f"//h6[@title='{titulo}']/ancestor::div[contains(@class,'css-kbfjxp')][1]"
                   f"//button[@data-testid='actions-toolbar-confirm']")
    ))
    driver.execute_script("arguments[0].click();", btn)
    time.sleep(0.5)


def click_descargar(driver, wait):
    btn = wait.until(EC.element_to_be_clickable(
        (By.XPATH, "//button[.//span[normalize-space()='Descargar'] or contains(., 'Descargar') or .//*[contains(@class,'lui-icon--download')]]"
                   " | //a[.//span[normalize-space()='Descargar'] or contains(., 'Descargar') or .//*[contains(@class,'lui-icon--download')]]")
    ))
    driver.execute_script("arguments[0].click();", btn)
    time.sleep(1)


def esperar_descarga_finalizada(download_dir: Path, timeout: int = 180) -> Path:
    end = time.time() + timeout
    before = {p.name for p in download_dir.glob("*")}

    while time.time() < end:
        current = list(download_dir.glob("*"))
        cr = list(download_dir.glob("*.crdownload"))

        nuevos = [p for p in current if p.name not in before and p.suffix.lower() == ".csv"]
        if nuevos and not cr:
            # devuelve el más nuevo
            return max(nuevos, key=lambda p: p.stat().st_mtime)

        time.sleep(0.5)

    raise TimeoutError(f"No se detectó descarga en {timeout}s en {download_dir}")


def mover_y_renombrar(archivo_descargado: Path, destino_dir: Path, final_name: str) -> Path:
    destino = destino_dir / final_name
    # Si ya existe, lo reemplaza
    if destino.exists():
        destino.unlink()
    archivo_descargado.rename(destino)
    return destino


# ---- Driver con carpeta de descarga fija ----
options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless=new")

prefs = {
    "download.default_directory": str(DOWNLOAD_DIR.resolve()),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)

try:
    driver.get(MASHUP_URL)
    wait.until(EC.presence_of_element_located((By.XPATH, "//div[@data-testid='collapsed-title-Año']")))

    try:
        wait.until(EC.invisibility_of_element_located((By.ID, "inline-overlay")))
    except Exception:
        pass

    # Año
    anio = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@data-testid='collapsed-title-Año']")))
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", anio)
    driver.execute_script("arguments[0].click();", anio)
    time.sleep(1)
    seleccionar_valores(driver, wait, valores=ANIOS, seleccionar_todos=SELECCIONAR_TODOS_ANIOS)
    confirmar_seleccion(driver, wait, "Año")
    time.sleep(1)

    # Mes
    mes = wait.until(EC.element_to_be_clickable((By.XPATH, "//div[@data-testid='collapsed-title-Mes']")))
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", mes)
    driver.execute_script("arguments[0].click();", mes)
    time.sleep(1)
    seleccionar_valores(driver, wait, valores=MESES_SELECCION, seleccionar_todos=SELECCIONAR_TODOS_MESES)
    confirmar_seleccion(driver, wait, "Mes")
    time.sleep(1)

    # Descargar + detectar archivo + renombrar
    click_descargar(driver, wait)
    descargado = esperar_descarga_finalizada(DOWNLOAD_DIR, timeout=1000)
    final_path = mover_y_renombrar(descargado, DOWNLOAD_DIR, FINAL_NAME)

    print("Descargado y guardado en:", final_path)

finally:
    driver.quit()


Descargado y guardado en: /Users/valentinauretadelafuente/Desktop/Operacion Real/Datos Operación/Generacion_real_2026_Feb.csv


In [2]:
# Fechas
from datetime import date
ini = date(2025, 1, 18)
fin = date(2025, 1, 31)

## Descarga documentos PRG/PO/TCO

In [2]:
from __future__ import annotations

import time
import zipfile
import shutil
from pathlib import Path
from datetime import date, timedelta
from typing import Optional, Tuple, Dict, List

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import ElementClickInterceptedException


URL_DOCS = "https://www.coordinador.cl/operacion/documentos/programas-de-operacion-2021/"


def asegurar_carpetas(base_dir: Path) -> Tuple[Path, Path, Path, Path]:
    programas_prg = base_dir / "Programas" / "PRG"
    programas_po = base_dir / "Programas" / "PO"
    tco_dir = base_dir / "TCO"
    tmp_download = base_dir / "_tmp_download"

    programas_prg.mkdir(parents=True, exist_ok=True)
    programas_po.mkdir(parents=True, exist_ok=True)
    tco_dir.mkdir(parents=True, exist_ok=True)
    tmp_download.mkdir(parents=True, exist_ok=True)

    return programas_prg, programas_po, tco_dir, tmp_download


def limpiar_tmp(tmp: Path, patterns: List[str]):
    for pat in patterns:
        for p in tmp.glob(pat):
            try:
                if p.is_file():
                    p.unlink()
            except Exception:
                pass


def esperar_archivo_sin_crdownload(
    carpeta: Path,
    nombre_objetivo: str,
    timeout_s: int = 120,
    poll_s: float = 1.0
) -> Optional[Path]:
    destino = carpeta / nombre_objetivo
    crdownload = carpeta / f"{nombre_objetivo}.crdownload"

    t0 = time.time()
    while time.time() - t0 < timeout_s:
        if destino.exists() and not crdownload.exists():
            return destino
        time.sleep(poll_s)

    return None

def make_driver(download_dir: Path) -> webdriver.Chrome:
    chrome_options = Options()
    chrome_options.add_argument("--start-minimized")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--safebrowsing-disable-download-protection")

    prefs = {
        "download.default_directory": str(download_dir.resolve()),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,

        # CLAVE: permitir múltiples descargas sin prompt
        "profile.default_content_setting_values.automatic_downloads": 1,
    }
    chrome_options.add_experimental_option("prefs", prefs)

    return webdriver.Chrome(options=chrome_options)


def set_fechas_y_buscar(driver: webdriver.Chrome, wait: WebDriverWait, fecha: date):
    fecha_inicio = (fecha - timedelta(days=1)).strftime("%Y-%m-%d")
    fecha_termino = (fecha + timedelta(days=1)).strftime("%Y-%m-%d")

    driver.execute_script(
        """
        const inicio = document.getElementById('fecha-val-inicio');
        const termino = document.getElementById('fecha-val-termino');
        if (!inicio || !termino) { throw new Error('No se encontraron inputs de fecha'); }

        inicio.value = arguments[0];
        inicio.dispatchEvent(new Event('change', { bubbles: true }));

        termino.value = arguments[1];
        termino.dispatchEvent(new Event('change', { bubbles: true }));
        """,
        fecha_inicio,
        fecha_termino
    )

    close_xpaths = [
        "//button[contains(.,'Continuar')] | //a[contains(.,'Continuar')]",
        "//button[contains(.,'Aceptar')] | //button[contains(.,'Entendido')] | //button[contains(.,'OK')]",
        "//*[@aria-label='Cerrar' or @aria-label='Close']",
        "//button[contains(@class,'close') or contains(@class,'btn-close')]",
    ]
    for xp in close_xpaths:
        try:
            el = WebDriverWait(driver, 2).until(EC.element_to_be_clickable((By.XPATH, xp)))
            driver.execute_script("arguments[0].click();", el)
            time.sleep(0.3)
        except Exception:
            pass

    boton = wait.until(EC.presence_of_element_located((By.XPATH, "//input[@value='Realizar Busqueda']")))
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", boton)
    time.sleep(0.2)

    driver.execute_script("""
      const killers = Array.from(document.querySelectorAll('div,section,aside'))
        .filter(el => {
          const s = getComputedStyle(el);
          const z = parseInt(s.zIndex || '0');
          return s.position === 'fixed' && z >= 1000 && el.offsetHeight > 50 && el.offsetWidth > 50;
        });
      killers.slice(0,5).forEach(el => el.style.display='none');
    """)

    try:
        wait.until(EC.element_to_be_clickable((By.XPATH, "//input[@value='Realizar Busqueda']"))).click()
    except (ElementClickInterceptedException, TimeoutException):
        driver.execute_script("arguments[0].click();", boton)

    try:
        ver_mas = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.ID, "readMore_btn")))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", ver_mas)
        time.sleep(0.2)
        try:
            ver_mas.click()
        except ElementClickInterceptedException:
            driver.execute_script("arguments[0].click();", ver_mas)
    except Exception:
        pass


def click_descarga_zip_por_nombre(driver: webdriver.Chrome, wait: WebDriverWait, nombre_zip: str) -> bool:
    enlaces_zip = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a[href$='.zip']")))
    enlace_obj = None
    for a in enlaces_zip:
        try:
            href = a.get_attribute("href") or ""
            if nombre_zip in href:
                enlace_obj = a
                break
        except Exception:
            continue

    if not enlace_obj:
        return False

    driver.execute_script("arguments[0].click();", enlace_obj)
    return True


def descargar_programa_y_extraer_prg_po(
    driver: webdriver.Chrome,
    wait: WebDriverWait,
    fecha: date,
    prg_dir: Path,
    po_dir: Path,
    tmp: Path
) -> Tuple[Optional[Path], Optional[Path]]:

    yyyymmdd = fecha.strftime("%Y%m%d")
    yymmdd = fecha.strftime("%y%m%d")

    salida_prg = prg_dir / f"PRG{yymmdd}.xlsx"
    salida_po = po_dir / f"PO{yymmdd}.xlsx"

    # Si ya están, no re-descarga
    if salida_prg.exists() and salida_po.exists():
        return salida_prg, salida_po

    limpiar_tmp(tmp, ["PROGRAMA*.zip", "PRG*.xlsx", "PO*.xlsx", "*.crdownload"])

    candidatos = [f"PROGRAMA{yyyymmdd}.zip", f"PROGRAMA{yyyymmdd}-1.zip", f"PROGRAMA{yyyymmdd}-2.zip"]

    elegido = None
    for nombre_zip in candidatos:
        ok = click_descarga_zip_por_nombre(driver, wait, nombre_zip)
        if ok:
            elegido = nombre_zip
            break

    if not elegido:
        return (salida_prg if salida_prg.exists() else None), (salida_po if salida_po.exists() else None)

    zip_path = esperar_archivo_sin_crdownload(tmp, elegido, timeout_s=180)
    if zip_path is None:
        return (salida_prg if salida_prg.exists() else None), (salida_po if salida_po.exists() else None)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(tmp)

    try:
        zip_path.unlink()
    except Exception:
        pass

    prg_src = tmp / f"PRG{yymmdd}.xlsx"
    po_src = tmp / f"PO{yymmdd}.xlsx"

    if not prg_src.exists():
        cands = list(tmp.glob(f"PRG{yymmdd}*.xlsx"))
        if cands:
            prg_src = cands[0]
    if not po_src.exists():
        cands = list(tmp.glob(f"PO{yymmdd}*.xlsx"))
        if cands:
            po_src = cands[0]

    prg_out = None
    po_out = None

    if prg_src.exists() and not salida_prg.exists():
        shutil.move(str(prg_src), str(salida_prg))
    if salida_prg.exists():
        prg_out = salida_prg

    if po_src.exists() and not salida_po.exists():
        shutil.move(str(po_src), str(salida_po))
    if salida_po.exists():
        po_out = salida_po

    for p in tmp.iterdir():
        if p.is_file():
            try:
                p.unlink()
            except Exception:
                pass

    return prg_out, po_out


def descargar_tco_y_extraer_xlsm(
    driver: webdriver.Chrome,
    wait: WebDriverWait,
    fecha: date,
    tco_dir: Path,
    tmp: Path
) -> Optional[Path]:

    yyyymmdd = fecha.strftime("%Y%m%d")

    salida = tco_dir / f"TCO{yyyymmdd}.xlsm"
    if salida.exists():
        return salida

    limpiar_tmp(tmp, ["TCO*.zip", "TCO*.xlsm", "*.crdownload"])

    zip_candidatos = [
        f"TCO{yyyymmdd}.zip",
        f"TCO{yyyymmdd}-1.zip",
        f"TCO{yyyymmdd}-2.zip",
    ]

    zip_elegido = None
    for nombre_zip in zip_candidatos:
        if click_descarga_zip_por_nombre(driver, wait, nombre_zip):
            zip_elegido = nombre_zip
            break

    if not zip_elegido:
        return None

    zip_path = esperar_archivo_sin_crdownload(tmp, zip_elegido, timeout_s=180)
    if zip_path is None:
        return None

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(tmp)

    try:
        zip_path.unlink()
    except Exception:
        pass

    xlsm_candidatos = [
        tmp / f"TCO{yyyymmdd}.xlsm",
        tmp / f"TCO{yyyymmdd}-1.xlsm",
        tmp / f"TCO{yyyymmdd}-2.xlsm",
    ]

    src = next((p for p in xlsm_candidatos if p.exists()), None)
    if src is None:
        cands = list(tmp.glob(f"TCO{yyyymmdd}*.xlsm"))
        if not cands:
            return None
        src = max(cands, key=lambda p: p.stat().st_mtime)

    shutil.move(str(src), str(salida))

    for p in tmp.iterdir():
        if p.is_file():
            try:
                p.unlink()
            except Exception:
                pass

    return salida


def descargar_prg_po_y_tco(
    fecha_inicial: date,
    fecha_final: Optional[date] = None,
    verbose: bool = True
) -> Tuple[Dict[date, str], Dict[date, str], Dict[date, str]]:
    base_dir = Path.cwd()
    prg_dir, po_dir, tco_dir, tmp = asegurar_carpetas(base_dir)

    if fecha_final is None:
        fecha_final = date.today()

    prg_out: Dict[date, str] = {}
    po_out: Dict[date, str] = {}
    tco_out: Dict[date, str] = {}

    driver = make_driver(tmp)
    wait = WebDriverWait(driver, 20)

    try:
        driver.get(URL_DOCS)

        f = fecha_inicial
        while f <= fecha_final:
            try:
                set_fechas_y_buscar(driver, wait, f)

                prg_path, po_path = descargar_programa_y_extraer_prg_po(driver, wait, f, prg_dir, po_dir, tmp)
                if prg_path:
                    prg_out[f] = str(prg_path)
                    if verbose:
                        print(f"[PRG] OK  {f} -> {Path(prg_path).name}")
                else:
                    if verbose:
                        print(f"[PRG] NO  {f}")

                if po_path:
                    po_out[f] = str(po_path)
                    if verbose:
                        print(f"[PO ] OK  {f} -> {Path(po_path).name}")
                else:
                    if verbose:
                        print(f"[PO ] NO  {f}")

                tco_path = descargar_tco_y_extraer_xlsm(driver, wait, f, tco_dir, tmp)
                if tco_path:
                    tco_out[f] = str(tco_path)
                    if verbose:
                        print(f"[TCO] OK  {f} -> {Path(tco_path).name}")
                else:
                    if verbose:
                        print(f"[TCO] NO  {f}")

            except TimeoutException as e:
                if verbose:
                    print(f"[ERR] {f} timeout: {e}")

            f += timedelta(days=1)

    finally:
        driver.quit()

    return prg_out, po_out, tco_out


if __name__ == "__main__":
    prg, po, tco = descargar_prg_po_y_tco(ini, fin, verbose=True)
    print(f"\nTotal PRG: {len(prg)}")
    print(f"Total PO : {len(po)}")
    print(f"Total TCO: {len(tco)}")


[PRG] NO  2026-02-23
[PO ] NO  2026-02-23
[TCO] NO  2026-02-23
[PRG] NO  2026-02-24
[PO ] NO  2026-02-24
[TCO] NO  2026-02-24
[PRG] NO  2026-02-25
[PO ] NO  2026-02-25
[TCO] NO  2026-02-25
[PRG] NO  2026-02-26
[PO ] NO  2026-02-26
[TCO] NO  2026-02-26
[PRG] NO  2026-02-27
[PO ] NO  2026-02-27
[TCO] NO  2026-02-27
[PRG] NO  2026-02-28
[PO ] NO  2026-02-28
[TCO] NO  2026-02-28

Total PRG: 0
Total PO : 0
Total TCO: 0


### Función para identificar a que Subtipo corresponden las centrales

In [1]:
import re
from datetime import date
from typing import Optional

def subtipo_termica_por_nombre(central: str) -> Optional[str]:
    c = central.upper()
    if "SAE" in c:
        return "Bess"
    if "PFV" in c:
        return "Solar"
    if "PE" in c:
        return "Eólica"
    if "DIESEL" in c:
        return "Diésel"
    if "GNL_INFLEX" in c: 
        return "GNL INF"
    if "GLP" in c: 
        return "GLP"
    if "GNL" in c:
        return "GNL"
    if "GN" in c:
        return "GNA"
    if "HFO" in c or "IFO" in c or "FO6" in c:
        return "Fuil Oil"
    if "GEO" in c:
        return "Geotermica"
    if "CAR" in c:
        return "Carbon"
    if "BIOGAS" in c or "SANTAMARTA" in c:
        return "BioGas"
    if "ENAPBIOBIO" in c:
        return "PetCoke"
    if "BIOMASA" in c or "COGEN" in c:
        return "Biomasa"
    if "PROPANO" in c:
        return "PROPANO"
    return None

## Lee los archivos

In [39]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, Tuple, Optional, List
from datetime import time
from openpyxl import load_workbook


def _resolver_archivo_con_sufijos(path: str | Path) -> Path:
    path = Path(path)
    if path.exists():
        return path

    parent = path.parent
    stem = path.stem
    ext = path.suffix

    candidatos = [
        parent / f"{stem}{ext}",
        parent / f"{stem}-1{ext}",
        parent / f"{stem}-2{ext}",
    ]
    existentes = [p for p in candidatos if p.exists()]
    if existentes:
        return max(existentes, key=lambda p: p.stat().st_mtime)

    globbed = list(parent.glob(f"{stem}*{ext}"))
    if globbed:
        return max(globbed, key=lambda p: p.stat().st_mtime)

    raise FileNotFoundError(f"No se encontró: {path.name} (ni variantes -1/-2) en {parent}")


# -------------------------------------------------------
# 1) PRG 
# -------------------------------------------------------
def leer_prg(prg_path: str | Path) -> Tuple[Dict, Dict, Dict, Dict]:
    prg_path = _resolver_archivo_con_sufijos(prg_path)

    tablas_energia = {
        "Hidroeléctricas de Pasada",
        "Eólicas",
        "Solares",
        "Centrales de concentración solar",
        "Térmicas",
        "Embalses y Reguladas",
    }
    otras_tablas = {
        "Costos Marginales",
        "Sistemas de Almacenamiento",
        "Sistemas de Almacenamiento - Carga RED",
        "Sistemas de Almacenamiento - Carga ERV",
        "Reducción de Renovable Estimada",
    }

    wb = load_workbook(prg_path, read_only=True, data_only=True)
    ws = wb["PROGRAMA"]

    datos_por_titulo: Dict[str, Dict[str, Any]] = {}

    rows = ws.iter_rows(min_row=15, values_only=True)
    fila = next(rows, None)

    while fila is not None:
        titulo_celda = fila[2]  # Columna C
        if titulo_celda is None or str(titulo_celda).strip() == "":
            fila = next(rows, None)
            continue

        titulo = str(titulo_celda).strip()
        def norm_title(x: str) -> str:
            # normaliza espacios raros y guiones
            x = x.replace("\u00A0", " ")          # NBSP -> espacio
            x = " ".join(x.split())              # colapsa espacios
            x = x.replace("–", "-").replace("—", "-")  # guiones raros -> "-"
            return x.strip()

        titulo = norm_title(str(titulo_celda))

        es_tabla_energia = titulo in tablas_energia
        es_costos = titulo == "Costos Marginales"
        es_sistemas = titulo.startswith("Sistemas de Almacenamiento")  # incluye Carga RED/ERV
        es_reduccion = titulo == "Reducción de Renovable Estimada"

        if not (es_tabla_energia or es_costos or es_sistemas or es_reduccion):
            fila = next(rows, None)
            continue

        fila = next(rows, None)
        centrales: Dict[str, Any] = {}

        while fila is not None:
            nombre = fila[2]  # Columna C
            if nombre is None or str(nombre).strip() == "":
                break

            nombre = str(nombre).strip()
            if nombre == "Total" and titulo != "Reducción de Renovable Estimada":
                fila = next(rows, None)
                continue

            codigo = fila[3]  # Columna D
            valores = []
            for i in range(24):
                v = fila[4 + i] if (4 + i) < len(fila) else None
                valores.append(None if v is None else round(float(v), 1))

            serie = [(time(h, 0, 0), valores[h]) for h in range(24)]

            if es_tabla_energia:
                centrales[nombre] = (codigo, titulo, serie)
            elif es_costos:
                centrales[nombre] = (codigo, serie)
            elif es_sistemas:
                centrales[nombre] = (codigo, serie)
            elif es_reduccion:
                centrales[nombre] = serie


            fila = next(rows, None)

        datos_por_titulo[titulo] = centrales
        fila = next(rows, None)

    dict_energia: Dict[str, Any] = {}
    for k in [
        "Hidroeléctricas de Pasada",
        "Eólicas",
        "Solares",
        "Centrales de concentración solar",
        "Térmicas",
        "Embalses y Reguladas",
    ]:
        dict_energia.update(datos_por_titulo.get(k, {}))

    dict_costos = datos_por_titulo.get("Costos Marginales", {})
    dict_reduccion = datos_por_titulo.get("Reducción de Renovable Estimada", {})
    dict_sistemas_red = {}
    dict_sistemas_erv = {}
    dict_sistemas_otro = {}

    for k, v in datos_por_titulo.items():
        kk = norm_title(k)
        if kk == "Sistemas de Almacenamiento - Carga RED":
            dict_sistemas_red = v
        elif kk == "Sistemas de Almacenamiento - Carga ERV":
            dict_sistemas_erv = v
        elif kk.startswith("Sistemas de Almacenamiento"):
            dict_sistemas_otro.update(v)

    return dict_energia, dict_costos, dict_sistemas_otro, dict_reduccion, dict_sistemas_red, dict_sistemas_erv

    



# -------------------------------------------------------
# 2) PO 
# -------------------------------------------------------
def leer_po_fijo(
    po_path: str | Path,
    sheet_name: str = "RESUMEN",
    first_data_row: int = 34
) -> List[Dict[str, Any]]:
    po_path = Path(po_path)
    wb = load_workbook(po_path, read_only=True, data_only=True)
    ws = wb[sheet_name] if sheet_name in wb.sheetnames else wb[wb.sheetnames[0]]

    def norm(x):
        if x is None:
            return ""
        return " ".join(str(x).replace("\n", " ").split()).strip()

    out: List[Dict[str, Any]] = []
    r = first_data_row

    while r <= (ws.max_row or r):
        central = norm(ws.cell(r, 2).value)  # col 2
        if central == "":
            break

        row = {
            "Central": central,
            "E_max_generable": ws.cell(r, 3).value,
            "Energia": ws.cell(r, 4).value,
            "Cmg_CVT": ws.cell(r, 5).value,
            "CVC": ws.cell(r, 6).value,
            "CVNC": ws.cell(r, 7).value,
            "CMed_Min_Tec": ws.cell(r, 8).value,
            "Combustible_Costo": ws.cell(r, 9).value,
            "Combustible_Unidad": ws.cell(r, 10).value,
        }
        out.append(row)
        r += 1

    return out


# -------------------------------------------------------
# 3) TCO 
# -------------------------------------------------------
def leer_tco_cv_fijo(
    tco_path: str | Path,
    sheet_name: str = "CV",
    header_row: int = 20,
) -> List[Dict[str, Any]]:

    tco_path = Path(tco_path)
    wb = load_workbook(tco_path, read_only=True, data_only=True)
    ws = wb[sheet_name] if sheet_name in wb.sheetnames else wb[wb.sheetnames[0]]

    COL_CENTRAL = 1
    COL_TIPO = 2
    COL_CV = 3
    COL_EF = 4  
    COL_BARRA = 5
    COL_NOMBRE_BARRA = 6
    COL_HORAS_START = 7   
    COL_BLOQUE1 = 31      
    COL_BLOQUE2 = 32
    COL_BLOQUE3 = 33

    def norm(x: Any) -> str:
        if x is None:
            return ""
        return " ".join(str(x).replace("\n", " ").split()).strip()

    out: List[Dict[str, Any]] = []
    r = header_row + 1

    while r <= (ws.max_row or r):
        central = norm(ws.cell(r, COL_CENTRAL).value)
        if central == "":
            break

        tipo_raw = norm(ws.cell(r, COL_TIPO).value)  
        tipo = None
        subtipo = None

        if tipo_raw.upper() == "E":
            tipo = "Embalse"
        elif tipo_raw.upper() == "T":
            tipo = "Termica"
            subtipo = subtipo_termica_por_nombre(central)
        else:
            tipo = None  

        row: Dict[str, Any] = {
            "Central": central,
            "Tipo": tipo,
            "Subtipo": subtipo,
            "CV": ws.cell(r, COL_CV).value,
            "Eficiencia": ws.cell(r, COL_EF).value,
            "Barra": ws.cell(r, COL_BARRA).value,
            "Nombre_Barra": ws.cell(r, COL_NOMBRE_BARRA).value,
            "Bloque1": ws.cell(r, COL_BLOQUE1).value,
            "Bloque2": ws.cell(r, COL_BLOQUE2).value,
            "Bloque3": ws.cell(r, COL_BLOQUE3).value,
        }

        for h in range(1, 25):
            row[f"Hora{h}"] = ws.cell(r, COL_HORAS_START + (h - 1)).value

        out.append(row)
        r += 1

    return out


## Creamos base de datos

## Creamos tabla y subimos Programas

In [38]:
# CELDA 1 — Cambios de esquema (ajusta PK/UNIQUE) + crear tablas faltantes

import mysql.connector

MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "rootvale123"
MYSQL_DB = "costos_variables"

def migrar_esquema(conn):
    cur = conn.cursor()

    # 1) Tabla energía: cambiar UNIQUE de (Fecha, Central) -> (Fecha, Central, Subtipo)
    #    (si ya existe, se ignora el error)
    cur.execute("""
    CREATE TABLE IF NOT EXISTS prg_energia_horaria (
      id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
      Fecha DATE NOT NULL,
      Central VARCHAR(255) NOT NULL,
      Categoria VARCHAR(80) NOT NULL,
      Subtipo VARCHAR(80) NULL,
      Hora1  DECIMAL(12,6) NULL, Hora2  DECIMAL(12,6) NULL, Hora3  DECIMAL(12,6) NULL, Hora4  DECIMAL(12,6) NULL,
      Hora5  DECIMAL(12,6) NULL, Hora6  DECIMAL(12,6) NULL, Hora7  DECIMAL(12,6) NULL, Hora8  DECIMAL(12,6) NULL,
      Hora9  DECIMAL(12,6) NULL, Hora10 DECIMAL(12,6) NULL, Hora11 DECIMAL(12,6) NULL, Hora12 DECIMAL(12,6) NULL,
      Hora13 DECIMAL(12,6) NULL, Hora14 DECIMAL(12,6) NULL, Hora15 DECIMAL(12,6) NULL, Hora16 DECIMAL(12,6) NULL,
      Hora17 DECIMAL(12,6) NULL, Hora18 DECIMAL(12,6) NULL, Hora19 DECIMAL(12,6) NULL, Hora20 DECIMAL(12,6) NULL,
      Hora21 DECIMAL(12,6) NULL, Hora22 DECIMAL(12,6) NULL, Hora23 DECIMAL(12,6) NULL, Hora24 DECIMAL(12,6) NULL,
      PRIMARY KEY (id),
      UNIQUE KEY uq_fecha_central_subtipo (Fecha, Central, Subtipo),
      KEY idx_categoria (Categoria),
      KEY idx_subtipo (Subtipo)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci;
    """)

    # Si ya existía uq_fecha_central, lo eliminamos y dejamos el nuevo
    try:
        cur.execute("ALTER TABLE prg_energia_horaria DROP INDEX uq_fecha_central;")
    except mysql.connector.Error:
        pass

    # Asegurar que el nuevo índice exista (si la tabla ya existía, CREATE TABLE no lo recrea)
    try:
        cur.execute("ALTER TABLE prg_energia_horaria ADD UNIQUE KEY uq_fecha_central_subtipo (Fecha, Central, Subtipo);")
    except mysql.connector.Error:
        pass

    # 2) Tabla costos marginales
    cur.execute("""
    CREATE TABLE IF NOT EXISTS costos_marginales (
      id BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
      Fecha DATE NOT NULL,
      Barra VARCHAR(255) NOT NULL,
      Codigo VARCHAR(80) NULL,
      Hora1  DECIMAL(12,6) NULL, Hora2  DECIMAL(12,6) NULL, Hora3  DECIMAL(12,6) NULL, Hora4  DECIMAL(12,6) NULL,
      Hora5  DECIMAL(12,6) NULL, Hora6  DECIMAL(12,6) NULL, Hora7  DECIMAL(12,6) NULL, Hora8  DECIMAL(12,6) NULL,
      Hora9  DECIMAL(12,6) NULL, Hora10 DECIMAL(12,6) NULL, Hora11 DECIMAL(12,6) NULL, Hora12 DECIMAL(12,6) NULL,
      Hora13 DECIMAL(12,6) NULL, Hora14 DECIMAL(12,6) NULL, Hora15 DECIMAL(12,6) NULL, Hora16 DECIMAL(12,6) NULL,
      Hora17 DECIMAL(12,6) NULL, Hora18 DECIMAL(12,6) NULL, Hora19 DECIMAL(12,6) NULL, Hora20 DECIMAL(12,6) NULL,
      Hora21 DECIMAL(12,6) NULL, Hora22 DECIMAL(12,6) NULL, Hora23 DECIMAL(12,6) NULL, Hora24 DECIMAL(12,6) NULL,
      PRIMARY KEY (id),
      UNIQUE KEY uq_fecha_barra (Fecha, Barra),
      KEY idx_barra (Barra)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci;
    """)

    conn.commit()
    cur.close()

conn = mysql.connector.connect(
    host=MYSQL_HOST, port=MYSQL_PORT,
    user=MYSQL_USER, password=MYSQL_PASSWORD,
    database=MYSQL_DB,
)
try:
    migrar_esquema(conn)
    print("OK esquema")
finally:
    conn.close()


OK esquema


In [40]:
# CELDA 2 — Upserts: energía + costos marginales + “Sistemas de Almacenamiento - Carga ...” a energía

from datetime import date
from typing import Any, Dict, Tuple

import mysql.connector

# Se asume que leer_prg(...) devuelve:
# dict_energia, dict_costos, dict_sistemas, dict_reduccion

SQL_UPSERT_ENERGIA = """
INSERT INTO prg_energia_horaria
(Fecha, Central, Categoria, Subtipo,
 Hora1, Hora2, Hora3, Hora4, Hora5, Hora6, Hora7, Hora8, Hora9, Hora10, Hora11, Hora12,
 Hora13, Hora14, Hora15, Hora16, Hora17, Hora18, Hora19, Hora20, Hora21, Hora22, Hora23, Hora24)
VALUES
(%s, %s, %s, %s,
 %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
 %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
  Categoria=VALUES(Categoria),
  Hora1=VALUES(Hora1), Hora2=VALUES(Hora2), Hora3=VALUES(Hora3), Hora4=VALUES(Hora4),
  Hora5=VALUES(Hora5), Hora6=VALUES(Hora6), Hora7=VALUES(Hora7), Hora8=VALUES(Hora8),
  Hora9=VALUES(Hora9), Hora10=VALUES(Hora10), Hora11=VALUES(Hora11), Hora12=VALUES(Hora12),
  Hora13=VALUES(Hora13), Hora14=VALUES(Hora14), Hora15=VALUES(Hora15), Hora16=VALUES(Hora16),
  Hora17=VALUES(Hora17), Hora18=VALUES(Hora18), Hora19=VALUES(Hora19), Hora20=VALUES(Hora20),
  Hora21=VALUES(Hora21), Hora22=VALUES(Hora22), Hora23=VALUES(Hora23), Hora24=VALUES(Hora24);
"""

SQL_UPSERT_CM = """
INSERT INTO costos_marginales
(Fecha, Barra, Codigo,
 Hora1, Hora2, Hora3, Hora4, Hora5, Hora6, Hora7, Hora8, Hora9, Hora10, Hora11, Hora12,
 Hora13, Hora14, Hora15, Hora16, Hora17, Hora18, Hora19, Hora20, Hora21, Hora22, Hora23, Hora24)
VALUES
(%s, %s, %s,
 %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
 %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
  Codigo=VALUES(Codigo),
  Hora1=VALUES(Hora1), Hora2=VALUES(Hora2), Hora3=VALUES(Hora3), Hora4=VALUES(Hora4),
  Hora5=VALUES(Hora5), Hora6=VALUES(Hora6), Hora7=VALUES(Hora7), Hora8=VALUES(Hora8),
  Hora9=VALUES(Hora9), Hora10=VALUES(Hora10), Hora11=VALUES(Hora11), Hora12=VALUES(Hora12),
  Hora13=VALUES(Hora13), Hora14=VALUES(Hora14), Hora15=VALUES(Hora15), Hora16=VALUES(Hora16),
  Hora17=VALUES(Hora17), Hora18=VALUES(Hora18), Hora19=VALUES(Hora19), Hora20=VALUES(Hora20),
  Hora21=VALUES(Hora21), Hora22=VALUES(Hora22), Hora23=VALUES(Hora23), Hora24=VALUES(Hora24);
"""

def serie_a_24(serie):
    vals = [v for (_t, v) in serie]
    if len(vals) != 24:
        vals = (vals + [None] * 24)[:24]
    return vals

def upsert_energia_dict(cur, fecha: date, dict_energia: Dict[str, Any]):
    for central, (_codigo, categoria, serie) in dict_energia.items():
        categoria = str(categoria)
        subtipo = None
        if categoria.strip().lower() in ("térmicas", "termicas"):
            subtipo = subtipo_termica_por_nombre(central)

        vals = serie_a_24(serie)
        cur.execute(SQL_UPSERT_ENERGIA, (fecha, central, categoria, subtipo, *vals))

def upsert_costos_marginales(cur, fecha: date, dict_costos: Dict[str, Any]):
    # dict_costos: {barra: (codigo, serie)}
    for barra, (codigo, serie) in dict_costos.items():
        vals = serie_a_24(serie)
        cur.execute(SQL_UPSERT_CM, (fecha, barra, codigo, *vals))

def upsert_sistemas_almacenamiento_a_energia(cur, fecha: date, dict_sistemas: Dict[str, Any]):
    """
    dict_sistemas: {nombre: (codigo, serie)}   (según tu leer_prg)
    Solo sube los que correspondan a:
      - "Sistemas de Almacenamiento - Carga RED" -> Categoria=Bess, Subtipo=Retiro
      - "Sistemas de Almacenamiento - Carga ERV" -> Categoria=Bess, Subtipo=Inyección
    Regla: si el nombre contiene esas frases.
    """
    for nombre, (codigo, serie) in dict_sistemas.items():
        n = str(nombre)

        if "Sistemas de Almacenamiento - Carga RED" in n:
            categoria = "Bess"
            subtipo = "Retiro"
        elif "Sistemas de Almacenamiento - Carga ERV" in n:
            categoria = "Bess"
            subtipo = "Inyección"
        else:
            continue

        # Guardamos en la tabla de energía:
        # Central = nombre (como viene en el PRG)
        vals = serie_a_24(serie)
        cur.execute(SQL_UPSERT_ENERGIA, (fecha, n, categoria, subtipo, *vals))

def subir_prg_completo(prg_path, fecha: date):
    dict_energia, dict_costos, dict_sistemas, dict_reduccion = leer_prg(prg_path)

    conn = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT,
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        database=MYSQL_DB,
    )
    try:
        cur = conn.cursor()
        upsert_energia_dict(cur, fecha, dict_energia)
        upsert_costos_marginales(cur, fecha, dict_costos)
        upsert_sistemas_almacenamiento_a_energia(cur, fecha, dict_sistemas)
        conn.commit()
        print("OK:", fecha, prg_path)
    finally:
        conn.close()


In [41]:
from datetime import date, timedelta
from pathlib import Path

def upsert_sae_carga_a_energia(cur, fecha: date, dict_red: dict, dict_erv: dict):
    # dict_red/dict_erv: {nombre: (codigo, serie)}
    for nombre, (codigo, serie) in dict_red.items():
        vals = serie_a_24(serie)
        cur.execute(SQL_UPSERT_ENERGIA, (fecha, str(nombre), "Bess", "Retiro", *vals))

    for nombre, (codigo, serie) in dict_erv.items():
        vals = serie_a_24(serie)
        cur.execute(SQL_UPSERT_ENERGIA, (fecha, str(nombre), "Bess", "Inyección", *vals))


def subir_prg_completo(prg_path, fecha: date):
    dict_energia, dict_costos, dict_sistemas, dict_reduccion, dict_red, dict_erv = leer_prg(prg_path)

    conn = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT,
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        database=MYSQL_DB,
    )
    try:
        cur = conn.cursor()
        upsert_energia_dict(cur, fecha, dict_energia)
        upsert_costos_marginales(cur, fecha, dict_costos)

        # NUEVO: RED/ERV
        upsert_sae_carga_a_energia(cur, fecha, dict_red, dict_erv)

        conn.commit()
        print("OK:", fecha, prg_path)
    finally:
        conn.close()


In [42]:
from datetime import date, timedelta
from pathlib import Path

def subir_prg_rango(folder_prg: str | Path, fecha_inicio: date, fecha_fin: date):
    """
    Sube todos los PRG entre fecha_inicio y fecha_fin (inclusive).
    Busca archivos tipo PRGYYMMDD.xlsx dentro de folder_prg.
    Usa tu _resolver_archivo_con_sufijos(...) dentro de leer_prg(prg_path).
    """
    folder_prg = Path(folder_prg)

    f = fecha_inicio
    while f <= fecha_fin:
        prg_name = f"PRG{f.strftime('%y%m%d')}.xlsx"
        prg_path = folder_prg / prg_name

        try:
            # subir_prg_completo llama leer_prg(...) que ya resuelve sufijos -1/-2
            subir_prg_completo(prg_path, f)
            print("[OK]", f, prg_path.name)
        except FileNotFoundError:
            print("[NO FILE]", f, prg_path.name)
        except Exception as e:
            print("[ERROR]", f, prg_path.name, "->", e)

        f += timedelta(days=1)

# USO:
subir_prg_rango("Programas/PRG", date(2026,3,3), date(2026,3,16))


OK: 2026-03-03 Programas/PRG/PRG260303.xlsx
[OK] 2026-03-03 PRG260303.xlsx
OK: 2026-03-04 Programas/PRG/PRG260304.xlsx
[OK] 2026-03-04 PRG260304.xlsx
OK: 2026-03-05 Programas/PRG/PRG260305.xlsx
[OK] 2026-03-05 PRG260305.xlsx
OK: 2026-03-06 Programas/PRG/PRG260306.xlsx
[OK] 2026-03-06 PRG260306.xlsx
OK: 2026-03-07 Programas/PRG/PRG260307.xlsx
[OK] 2026-03-07 PRG260307.xlsx
OK: 2026-03-08 Programas/PRG/PRG260308.xlsx
[OK] 2026-03-08 PRG260308.xlsx
OK: 2026-03-09 Programas/PRG/PRG260309.xlsx
[OK] 2026-03-09 PRG260309.xlsx
OK: 2026-03-10 Programas/PRG/PRG260310.xlsx
[OK] 2026-03-10 PRG260310.xlsx
OK: 2026-03-11 Programas/PRG/PRG260311.xlsx
[OK] 2026-03-11 PRG260311.xlsx
OK: 2026-03-12 Programas/PRG/PRG260312.xlsx
[OK] 2026-03-12 PRG260312.xlsx
OK: 2026-03-13 Programas/PRG/PRG260313.xlsx
[OK] 2026-03-13 PRG260313.xlsx
OK: 2026-03-14 Programas/PRG/PRG260314.xlsx
[OK] 2026-03-14 PRG260314.xlsx
OK: 2026-03-15 Programas/PRG/PRG260315.xlsx
[OK] 2026-03-15 PRG260315.xlsx
OK: 2026-03-16 Programas/

In [43]:
import mysql.connector

def corregir_subtipo_por_categoria_eliminando_conflictos():
    conn = mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DB,
    )
    try:
        cur = conn.cursor()

        # Mapeo Categoria -> Subtipo (ajusta strings exactamente a como están en tu tabla)
        mappings = [
            ("Hidroelectricas de Pasada", "Pasada"),
            ("Embalses y Reguladas",      "Embalse"),
            ("Eólicas",                   "Eólica"),
            ("Solares",                   "Solar"),
            ("Centrales de concentración solar","CS")
        ]

        conn.start_transaction()

        total_deleted = 0
        total_updated = 0

        # 1) Borrar filas que causarían duplicado al actualizar
        for categoria, subtipo_obj in mappings:
            # Borra SOLO las filas con Subtipo NULL/'' que chocarían con otra fila existente
            delete_sql = """
            DELETE a
            FROM costos_variables.prg_energia_horaria a
            JOIN costos_variables.prg_energia_horaria b
              ON b.Fecha = a.Fecha
             AND b.Central = a.Central
             AND b.Subtipo = %s
            WHERE a.Categoria = %s
              AND (a.Subtipo IS NULL OR a.Subtipo = '')
              AND a.id <> b.id;
            """
            cur.execute(delete_sql, (subtipo_obj, categoria))
            total_deleted += cur.rowcount

        # 2) Actualizar Subtipo
        update_sql = """
        UPDATE costos_variables.prg_energia_horaria
        SET Subtipo = CASE
            WHEN Categoria = 'Hidroelectricas de Pasada' THEN 'Pasada'
            WHEN Categoria = 'Embalses y Reguladas'      THEN 'Embalse'
            WHEN Categoria = 'Eólicas'                   THEN 'Eólica'
            WHEN Categoria = 'Solares'                   THEN 'Solar'
            WHEN Categoria = 'Centrales de concentración solar' THEN 'CS'
            ELSE Subtipo
        END
        WHERE Categoria IN ('Hidroelectricas de Pasada', 'Embalses y Reguladas', 'Eólicas', 'Solares','Centrales de concentración solar')
          AND (Subtipo IS NULL OR Subtipo = '');
        """
        cur.execute(update_sql)
        total_updated = cur.rowcount

        conn.commit()
        print("Filas borradas por conflicto de llave:", total_deleted)
        print("Filas actualizadas:", total_updated)

    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

corregir_subtipo_por_categoria_eliminando_conflictos()

Filas borradas por conflicto de llave: 997
Filas actualizadas: 12961


## Tabla barras planificación

In [44]:
import mysql.connector

MYSQL_HOST = "127.0.0.1"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "rootvale123"

DB_OR   = "costos_variables"
DB_OUT  = "planificacion"

TABLE_OR  = f"{DB_OR}.costos_marginales"   # (Fecha, Barra, Codigo, Hora1..Hora24)
TABLE_OUT = f"{DB_OUT}.barras"            # (Fecha, hora, A.JAHUEL, ...)

# Mapeo: nombre columna destino -> valor en costos_marginales.Barra
BARRAS_MAP = {
    "A.JAHUEL":      "AJahuel220",
    "CRUCERO":       "Crucero220",
    "MAITENCILLO":   "Maitencillo220",
    "P.MONTT":       "PMontt220",
    "QUILLOTA":      "Quillota220",
    "CHARRUA":       "Charrua220",
    "D.ALMAGRO":     "DAlmagro220",
}

def q_ident(s: str) -> str:
    return "`" + s.replace("`", "") + "`"

def q_literal(s: str) -> str:
    return s.replace("'", "''")

def ensure_out_table(cur):
    cols = ",\n  ".join([f"{q_ident(k)} DECIMAL(18,6) NULL" for k in BARRAS_MAP.keys()])
    cur.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_OUT}`;")
    cur.execute(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_OUT} (
      Fecha DATE NOT NULL,
      hora  TINYINT NOT NULL,
      {cols},
      PRIMARY KEY (Fecha, hora)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900_ai_ci;
    """)
    # por si agregas barras nuevas después
    for k in BARRAS_MAP.keys():
        try:
            cur.execute(f"ALTER TABLE {TABLE_OUT} ADD COLUMN {q_ident(k)} DECIMAL(18,6) NULL;")
        except mysql.connector.Error:
            pass

def upsert_barras_from_wide_hours(fecha_inicio, fecha_fin):
    """
    Convierte costos_marginales (Hora1..Hora24) a planificacion.barras (hora 0..23).
    """
    conn = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT,
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        database=DB_OR,
    )
    try:
        cur = conn.cursor()
        ensure_out_table(cur)

        # 1) UNPIVOT horas: Hora1..Hora24 -> filas (Fecha,hora,Barra,valor)
        #    Hora1 -> hora=0, ..., Hora24 -> hora=23
        unpivot_parts = []
        for h in range(1, 25):
            hora_out = h - 1
            unpivot_parts.append(f"""
            SELECT
              o.Fecha,
              {hora_out} AS hora,
              o.Barra,
              o.{q_ident(f"Hora{h}")} AS valor
            FROM {TABLE_OR} o
            WHERE o.Fecha BETWEEN %s AND %s
              AND o.Barra IN ({", ".join(["%s"]*len(BARRAS_MAP))})
            """.strip())
        unpivot_sql = "\nUNION ALL\n".join(unpivot_parts)

        # 2) PIVOT a columnas por barra destino
        cols_insert = ["Fecha", "hora"] + [q_ident(k) for k in BARRAS_MAP.keys()]
        sums = []
        updates = []

        for col_out, barra_src in BARRAS_MAP.items():
            sums.append(
                f"MAX(CASE WHEN u.Barra = '{q_literal(barra_src)}' THEN u.valor ELSE NULL END) AS {q_ident(col_out)}"
            )
            updates.append(f"{q_ident(col_out)}=VALUES({q_ident(col_out)})")

        sql = f"""
        INSERT INTO {TABLE_OUT} ({", ".join(cols_insert)})
        SELECT
          u.Fecha,
          u.hora,
          {", ".join(sums)}
        FROM (
          {unpivot_sql}
        ) u
        GROUP BY u.Fecha, u.hora
        ON DUPLICATE KEY UPDATE
          {", ".join(updates)};
        """

        # parámetros: por cada SELECT del union => (fecha_ini, fecha_fin, *barras)
        barras_params = list(BARRAS_MAP.values())
        params = []
        for _ in range(24):
            params.extend([fecha_inicio, fecha_fin, *barras_params])

        cur.execute(sql, tuple(params))
        conn.commit()

        cur.execute(f"SELECT COUNT(*) FROM {TABLE_OUT} WHERE Fecha BETWEEN %s AND %s;", (fecha_inicio, fecha_fin))
        print("Filas en planificacion.barras (rango):", cur.fetchone()[0])

    finally:
        conn.close()

# USO:
from datetime import date
upsert_barras_from_wide_hours(date(2026,2,24), date(2026,3,16))


Filas en planificacion.barras (rango): 504


### Corregimos anteriores

## Tabla para PO

In [45]:
def _to_float_or_none(x):
    if x is None:
        return None
    try:
        if isinstance(x, str):
            s = x.strip()
            if s.count(",") == 1 and s.count(".") >= 1:
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", ".")
            return float(s)
        return float(x)
    except Exception:
        return None


def upsert_po_termicas(conn, fecha, po_rows):
    sql = """
    INSERT INTO costos_variables.po
    (Fecha, Central, Categoria, Subtipo,
     E_max_generable, Energia, Cmg_CVT, CVC, CVNC, CMed_Min_Tec,
     Combustible_Costo, Combustible_Unidad)
    VALUES
    (%s, %s, %s, %s,
     %s, %s, %s, %s, %s, %s,
     %s, %s)
    ON DUPLICATE KEY UPDATE
      Categoria=VALUES(Categoria),
      Subtipo=VALUES(Subtipo),
      E_max_generable=VALUES(E_max_generable),
      Energia=VALUES(Energia),
      Cmg_CVT=VALUES(Cmg_CVT),
      CVC=VALUES(CVC),
      CVNC=VALUES(CVNC),
      CMed_Min_Tec=VALUES(CMed_Min_Tec),
      Combustible_Costo=VALUES(Combustible_Costo),
      Combustible_Unidad=VALUES(Combustible_Unidad);
    """

    params_list = []
    for r in po_rows:
        central = r.get("Central")
        if central is None or str(central).strip() == "":
            continue

        central_txt = str(central).strip()
        categoria = "Termicas"
        subtipo = subtipo_termica_por_nombre(central_txt)

        params_list.append((
            fecha,
            central_txt,
            categoria,
            subtipo,
            _to_float_or_none(r.get("E_max_generable")),
            _to_float_or_none(r.get("Energia")),
            _to_float_or_none(r.get("Cmg_CVT")),
            _to_float_or_none(r.get("CVC")),
            _to_float_or_none(r.get("CVNC")),
            _to_float_or_none(r.get("CMed_Min_Tec")),
            _to_float_or_none(r.get("Combustible_Costo")),
            None if r.get("Combustible_Unidad") in (None, "") else str(r.get("Combustible_Unidad")).strip(),
        ))

    if not params_list:
        return 0

    cur = conn.cursor()
    cur.executemany(sql, params_list)  
    conn.commit()                      
    cur.close()
    return len(params_list)


In [46]:
# Fechas
from datetime import date
ini = date(2026, 3, 3)
fin = date(2026, 3, 16)

In [47]:
from pathlib import Path
from datetime import timedelta
import time as _t
import mysql.connector

conn = mysql.connector.connect(
    host="127.0.0.1", port=3306, user="root", password="rootvale123",
    database="costos_variables"
)

po_dir = Path("Programas/PO")

f = ini
while f <= fin:
    po_path = po_dir / f"PO{f.strftime('%y%m%d')}.xlsx"

    if not po_path.exists():
        print(f"[NO FILE] {f} -> {po_path.name}")
        f += timedelta(days=1)
        continue

    try:
        t0 = _t.time()
        po_rows = leer_po_fijo(po_path)
        t1 = _t.time()

        n = upsert_po_termicas(conn, f, po_rows)
        t2 = _t.time()

        print(
            f"[OK] {f} -> {po_path.name} ({len(po_rows)} filas) | "
            f"leer: {t1 - t0:.2f}s | db: {t2 - t1:.2f}s | total: {t2 - t0:.2f}s"
        )

    except Exception as e:
        print(f"[ERROR] {f} -> {po_path.name}: {e}")

    f += timedelta(days=1)

conn.close()


[OK] 2026-03-03 -> PO260303.xlsx (692 filas) | leer: 71.15s | db: 0.12s | total: 71.27s
[OK] 2026-03-04 -> PO260304.xlsx (692 filas) | leer: 71.54s | db: 0.04s | total: 71.58s
[OK] 2026-03-05 -> PO260305.xlsx (692 filas) | leer: 71.60s | db: 0.04s | total: 71.64s
[OK] 2026-03-06 -> PO260306.xlsx (692 filas) | leer: 71.55s | db: 0.04s | total: 71.59s
[OK] 2026-03-07 -> PO260307.xlsx (692 filas) | leer: 71.59s | db: 0.04s | total: 71.63s
[OK] 2026-03-08 -> PO260308.xlsx (692 filas) | leer: 107.47s | db: 0.04s | total: 107.51s
[OK] 2026-03-09 -> PO260309.xlsx (692 filas) | leer: 71.62s | db: 0.04s | total: 71.66s
[OK] 2026-03-10 -> PO260310.xlsx (692 filas) | leer: 71.90s | db: 0.04s | total: 71.94s
[OK] 2026-03-11 -> PO260311.xlsx (692 filas) | leer: 71.66s | db: 0.04s | total: 71.70s
[OK] 2026-03-12 -> PO260312.xlsx (692 filas) | leer: 973.47s | db: 0.04s | total: 973.51s
[OK] 2026-03-13 -> PO260313.xlsx (692 filas) | leer: 1397.12s | db: 0.06s | total: 1397.18s
[OK] 2026-03-14 -> PO260

## Tabla para TCO

In [48]:
def _to_float_or_none(x):
    if x is None:
        return None
    try:
        if isinstance(x, str):
            s = x.strip()
            if s.count(",") == 1 and s.count(".") >= 1:
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", ".")
            return float(s)
        return float(x)
    except Exception:
        return None


def upsert_tco(conn, fecha: date, rows: list[dict]):
    sql = """
    INSERT INTO costos_variables.tco
    (Fecha, Central, Tipo, Subtipo, CV, Eficiencia, Barra, Nombre_Barra,
     Hora1, Hora2, Hora3, Hora4, Hora5, Hora6, Hora7, Hora8, Hora9, Hora10, Hora11, Hora12,
     Hora13, Hora14, Hora15, Hora16, Hora17, Hora18, Hora19, Hora20, Hora21, Hora22, Hora23, Hora24,
     Bloque1, Bloque2, Bloque3)
    VALUES
    (%s, %s, %s, %s, %s, %s, %s, %s,
     %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
     %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
     %s, %s, %s)
    ON DUPLICATE KEY UPDATE
      Tipo=VALUES(Tipo),
      Subtipo=VALUES(Subtipo),
      CV=VALUES(CV),
      Eficiencia=VALUES(Eficiencia),
      Barra=VALUES(Barra),
      Nombre_Barra=VALUES(Nombre_Barra),
      Hora1=VALUES(Hora1), Hora2=VALUES(Hora2), Hora3=VALUES(Hora3), Hora4=VALUES(Hora4),
      Hora5=VALUES(Hora5), Hora6=VALUES(Hora6), Hora7=VALUES(Hora7), Hora8=VALUES(Hora8),
      Hora9=VALUES(Hora9), Hora10=VALUES(Hora10), Hora11=VALUES(Hora11), Hora12=VALUES(Hora12),
      Hora13=VALUES(Hora13), Hora14=VALUES(Hora14), Hora15=VALUES(Hora15), Hora16=VALUES(Hora16),
      Hora17=VALUES(Hora17), Hora18=VALUES(Hora18), Hora19=VALUES(Hora19), Hora20=VALUES(Hora20),
      Hora21=VALUES(Hora21), Hora22=VALUES(Hora22), Hora23=VALUES(Hora23), Hora24=VALUES(Hora24),
      Bloque1=VALUES(Bloque1), Bloque2=VALUES(Bloque2), Bloque3=VALUES(Bloque3);
    """

    params_list = []
    for r in rows:
        params_list.append((
            fecha,
            r["Central"],
            r.get("Tipo"),
            r.get("Subtipo"),
            _to_float_or_none(r.get("CV")),
            _to_float_or_none(r.get("Eficiencia")),
            None if r.get("Barra") is None else str(r.get("Barra")).strip(),
            None if r.get("Nombre_Barra") is None else str(r.get("Nombre_Barra")).strip(),
            *[_to_float_or_none(r.get(f"Hora{i}")) for i in range(1, 25)],
            _to_float_or_none(r.get("Bloque1")),
            _to_float_or_none(r.get("Bloque2")),
            _to_float_or_none(r.get("Bloque3")),
        ))

    cur = conn.cursor()
    cur.executemany(sql, params_list)
    conn.commit()          
    cur.close()



In [49]:
from pathlib import Path
from typing import Any, Dict, List, Optional
from openpyxl import load_workbook

def leer_tco_cv_fijo_rapido(
    tco_path: str | Path,
    sheet_name: str = "CV",
    header_row: int = 20,
) -> List[Dict[str, Any]]:

    tco_path = Path(tco_path)

    wb = load_workbook(
        tco_path,
        read_only=True,
        data_only=True,
        keep_vba=False,   # importante: no necesitas macros para leer datos
        keep_links=False  # evita cargar vínculos
    )
    ws = wb[sheet_name] if sheet_name in wb.sheetnames else wb[wb.sheetnames[0]]

    def norm(x: Any) -> str:
        if x is None:
            return ""
        return " ".join(str(x).replace("\n", " ").split()).strip()

    out: List[Dict[str, Any]] = []

    for row in ws.iter_rows(
        min_row=header_row + 1,
        min_col=1,
        max_col=33,
        values_only=True
    ):
        central = norm(row[0])
        if central == "":
            break

        tipo_raw = norm(row[1]).upper()  # E/T
        if tipo_raw == "E":
            tipo = "Embalse"
            subtipo = None
        elif tipo_raw == "T":
            tipo = "Termica"
            subtipo = subtipo_termica_por_nombre(central)
        else:
            tipo = None
            subtipo = None

        d: Dict[str, Any] = {
            "Central": central,
            "Tipo": tipo,
            "Subtipo": subtipo,
            "CV": row[2],
            "Eficiencia": row[3],
            "Barra": row[4],
            "Nombre_Barra": row[5],
            "Bloque1": row[30],
            "Bloque2": row[31],
            "Bloque3": row[32],
        }

        for i in range(24):
            d[f"Hora{i+1}"] = row[6 + i]

        out.append(d)

    wb.close()
    return out


In [50]:
import time as _t
from datetime import date, timedelta
from pathlib import Path


tco_dir = Path("TCO")
conn = mysql.connector.connect(
    host="127.0.0.1", port=3306, user="root", password="rootvale123",
    database="costos_variables"
)

f = ini
while f <= fin:
    base = f"TCO{f.strftime('%Y%m%d')}"
    candidatos = [
        tco_dir / f"{base}.xlsm",
        tco_dir / f"{base}-1.xlsm",
        tco_dir / f"{base}-2.xlsm",
    ]

    tco_path = next((p for p in candidatos if p.exists()), None)

    if not tco_path:
        print(f"[NO FILE] {f} -> {base}.xlsm (-1/-2)")
        f += timedelta(days=1)
        continue

    try:
        t0 = _t.time()
        rows = leer_tco_cv_fijo_rapido(tco_path, sheet_name="CV", header_row=20)
        t1 = _t.time()

        upsert_tco(conn, f, rows)  # commitea 1 vez por archivo
        t2 = _t.time()

        print(
            f"[OK] {f} -> {tco_path.name} ({len(rows)} filas) | "
            f"leer: {t1 - t0:.2f}s | db: {t2 - t1:.2f}s | total: {t2 - t0:.2f}s"
        )

    except Exception as e:
        print(f"[ERROR] {f} -> {tco_path.name}: {e}")

    f += timedelta(days=1)



[OK] 2026-03-03 -> TCO20260303.xlsm (814 filas) | leer: 0.22s | db: 0.10s | total: 0.32s
[OK] 2026-03-04 -> TCO20260304.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-05 -> TCO20260305.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-06 -> TCO20260306.xlsm (814 filas) | leer: 0.27s | db: 0.06s | total: 0.33s
[OK] 2026-03-07 -> TCO20260307.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-08 -> TCO20260308.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-09 -> TCO20260309.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-10 -> TCO20260310.xlsm (814 filas) | leer: 0.26s | db: 0.06s | total: 0.31s
[OK] 2026-03-11 -> TCO20260311.xlsm (814 filas) | leer: 0.21s | db: 0.07s | total: 0.27s
[OK] 2026-03-12 -> TCO20260312.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-13 -> TCO20260313.xlsm (814 filas) | leer: 0.21s | db: 0.06s | total: 0.27s
[OK] 2026-03-14 -> TC

## Tabla reservas

In [51]:
from __future__ import annotations

from pathlib import Path
from datetime import date
from typing import Dict, Tuple, List, Optional

import mysql.connector
from openpyxl import load_workbook


# =========================
# Config MySQL destino
# =========================
MYSQL = dict(
    host="127.0.0.1",
    port=3306,
    user="root",
    password="rootvale123",
)

DB_DEST = "planificacion"
TABLE_DEST = "reserva_conf"  # planificacion.reserva_conf


# =========================
# Util: resolver PRG con sufijos
# =========================
def _resolver_archivo_con_sufijos(path: str | Path) -> Path:
    path = Path(path)
    if path.exists():
        return path

    parent = path.parent
    stem = path.stem
    ext = path.suffix

    candidatos = [
        parent / f"{stem}{ext}",
        parent / f"{stem}-1{ext}",
        parent / f"{stem}-2{ext}",
    ]
    existentes = [p for p in candidatos if p.exists()]
    if existentes:
        return max(existentes, key=lambda p: p.stat().st_mtime)

    globbed = list(parent.glob(f"{stem}*{ext}"))
    if globbed:
        return max(globbed, key=lambda p: p.stat().st_mtime)

    raise FileNotFoundError(f"No se encontró: {path.name} (ni variantes -1/-2) en {parent}")


def _norm(x) -> str:
    if x is None:
        return ""
    s = str(x).replace("\u00A0", " ")
    s = " ".join(s.split())
    return s.strip()


# =========================
# 1) Fecha desde nombre PRG (YYMMDD -> 20YY-MM-DD)
# =========================
def fecha_desde_nombre_prg(prg_path: str | Path) -> date:
    """
    Extrae YYMMDD desde el nombre (ej: PRG250101.xlsx -> 2025-01-01).
    Toma los últimos 6 dígitos consecutivos del stem.
    """
    p = Path(prg_path)
    stem = p.stem  # PRG250101 o PRG250101-1
    digits = "".join(ch for ch in stem if ch.isdigit())
    if len(digits) < 6:
        raise ValueError(f"No pude extraer YYMMDD desde el nombre: {p.name}")
    yymmdd = digits[-6:]
    yy = int(yymmdd[0:2])
    mm = int(yymmdd[2:4])
    dd = int(yymmdd[4:6])
    return date(2000 + yy, mm, dd)


# =========================
# 2) Lectura rápida "* - Conf"
#    (evita ws.cell por celda: usa iter_rows con values_only)
# =========================
def _find_in_rows(
    rows: List[Tuple],
    needle: str,
    *,
    min_r: int,
    min_c: int,
    max_r: int,
    max_c: int
) -> Optional[Tuple[int, int]]:
    """
    Busca needle exacto (case-insensitive) en la subgrilla.
    rows es lista de tuplas 0-indexed.
    Retorna (r, c) 0-indexed.
    """
    needle_n = _norm(needle).lower()
    R = min(len(rows), max_r)
    for r in range(min_r, R):
        row = rows[r]
        C = min(len(row), max_c)
        for c in range(min_c, C):
            if _norm(row[c]).lower() == needle_n:
                return r, c
    return None


def leer_conf_rapido(
    prg_path: str | Path,
    sheet_name: str,
    tipo_reserva: str,
    *,
    start_row: int = 6,   # fila excel (1-indexed) donde está "CONFIGURACIÓN"
    start_col: int = 7,   # col excel (1-indexed) G
    n_horas: int = 24
) -> List[Tuple[date, int, str, str, float, float]]:
    """
    Lee hoja "* - Conf" y devuelve:
      (Fecha, hora(0-23), Configuracion, TipoReserva, Subida, Bajada)
    La fecha se obtiene del nombre del archivo (YYMMDD).
    """

    prg_path = _resolver_archivo_con_sufijos(prg_path)
    fecha = fecha_desde_nombre_prg(prg_path)

    wb = load_workbook(prg_path, read_only=True, data_only=True)
    if sheet_name not in wb.sheetnames:
        raise ValueError(f"{prg_path.name} no tiene hoja '{sheet_name}'. Hojas: {wb.sheetnames}")
    ws = wb[sheet_name]

    # Leer solo un área razonable para velocidad.
    # Ajusta si alguna hoja viene más grande.
    max_row = min(ws.max_row, 600)
    max_col = min(ws.max_column, 220)

    rows = list(ws.iter_rows(min_row=1, max_row=max_row, min_col=1, max_col=max_col, values_only=True))

    # Convertir start_row/start_col a 0-index
    sr = max(start_row - 1, 0)
    sc = max(start_col - 1, 0)

    # Encontrar CONFIGURACIÓN cerca del header
    pos_cfg = _find_in_rows(rows, "CONFIGURACIÓN", min_r=sr, min_c=sc, max_r=sr + 30, max_c=sc + 30)
    if pos_cfg:
        header_r, cfg_c = pos_cfg
    else:
        header_r, cfg_c = sr, sc  # fallback: asumir (fila 6, col G)

    # Encontrar SUBIDA / BAJADA (buscamos en las primeras ~80 filas desde col G)
    pos_sub = _find_in_rows(rows, "SUBIDA", min_r=0, min_c=sc, max_r=min(80, len(rows)), max_c=max_col)
    pos_baj = _find_in_rows(rows, "BAJADA", min_r=0, min_c=sc, max_r=min(80, len(rows)), max_c=max_col)
    if not pos_sub or not pos_baj:
        raise ValueError(f"No encontré 'SUBIDA' y/o 'BAJADA' en '{sheet_name}' (desde col {start_col}).")

    sub_r, sub_c0 = pos_sub
    baj_r, baj_c0 = pos_baj

    # Fila de horas suele ser 1 debajo del título
    horas_row_sub = sub_r + 1
    horas_row_baj = baj_r + 1

    def build_hour_map(row_idx: int, col_start_idx: int) -> Dict[int, int]:
        m: Dict[int, int] = {}
        if row_idx >= len(rows):
            return {h: col_start_idx + (h - 1) for h in range(1, n_horas + 1)}
        row = rows[row_idx]
        for j in range(n_horas):
            c = col_start_idx + j
            if c >= len(row):
                continue
            v = row[c]
            try:
                h = int(str(v).strip())
            except Exception:
                continue
            if 1 <= h <= n_horas:
                m[h] = c
        if len(m) < n_horas:
            m = {h: col_start_idx + (h - 1) for h in range(1, n_horas + 1)}
        return m

    sub_hmap = build_hour_map(horas_row_sub, sub_c0)
    baj_hmap = build_hour_map(horas_row_baj, baj_c0)

    def to_float(x) -> float:
        if x is None or _norm(x) == "":
            return 0.0
        try:
            return float(x)
        except Exception:
            return 0.0

    out: List[Tuple[date, int, str, str, float, float]] = []

    data_start = header_r + 1  # datos bajo "CONFIGURACIÓN"
    empty_streak = 0

    for r in range(data_start, len(rows)):
        row = rows[r]
        cfg = _norm(row[cfg_c] if cfg_c < len(row) else None)

        if not cfg:
            empty_streak += 1
            if empty_streak >= 5:
                break
            continue
        empty_streak = 0

        for h in range(1, n_horas + 1):
            cs = sub_hmap[h]
            cb = baj_hmap[h]
            v_sub = row[cs] if cs < len(row) else None
            v_baj = row[cb] if cb < len(row) else None
            out.append((fecha, h - 1, cfg, tipo_reserva, to_float(v_sub), to_float(v_baj)))

    return out


# =========================
# 3) SQL: tabla planificacion.reserva_conf + upsert
# =========================
def ensure_table_reserva_conf(conn):
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_DEST}`;")
    cur.execute(f"""
    CREATE TABLE IF NOT EXISTS `{DB_DEST}`.`{TABLE_DEST}` (
      Fecha        DATE NOT NULL,
      hora         TINYINT NOT NULL,         -- 0..23
      Central VARCHAR(255) NOT NULL,
      TipoReserva  VARCHAR(20) NOT NULL,     -- Primaria/Secundaria/Terciaria
      SUBIDA       DECIMAL(18,6) NULL,
      BAJADA       DECIMAL(18,6) NULL,
      PRIMARY KEY (Fecha, hora, Configuracion, TipoReserva)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)
    conn.commit()


def upsert_reserva_conf(prg_path: str | Path):
    prg_path = _resolver_archivo_con_sufijos(prg_path)

    # hojas -> tipo
    targets = [
        ("Reservas CPF - Conf", "Primaria", 6, 7),
        ("Reservas CSF - Conf", "Secundaria",6, 2),
        ("Reservas CTF - Conf", "Terciaria",6, 2),
    ]

    all_rows: List[Tuple[date, int, str, str, float, float]] = []
    for sheet, tipo, sr, sc in targets:
        all_rows.extend(
            leer_conf_rapido(
                prg_path,
                sheet,
                tipo,
                start_row=sr,
                start_col=sc,
                n_horas=24
            )
        )
    if not all_rows:
        print("Sin filas para insertar.")
        return

    conn = mysql.connector.connect(**MYSQL)
    try:
        ensure_table_reserva_conf(conn)
        cur = conn.cursor()

        sql = f"""
        INSERT INTO `{DB_DEST}`.`{TABLE_DEST}`
          (Fecha, hora, Central, TipoReserva, SUBIDA, BAJADA)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
          SUBIDA = VALUES(SUBIDA),
          BAJADA = VALUES(BAJADA);
        """
        cur.executemany(sql, all_rows)
        conn.commit()

        print(f"OK: upsert {len(all_rows)} filas en {DB_DEST}.{TABLE_DEST} ({Path(prg_path).name})")
    finally:
        conn.close()


# =========================
# Uso
# =========================
upsert_reserva_conf("Programas/PRG/PRG250101.xlsx")


OK: upsert 30072 filas en planificacion.reserva_conf (PRG250101.xlsx)


In [52]:
from datetime import date, timedelta
from pathlib import Path

def _daterange(d1: date, d2: date):
    d = d1
    while d <= d2:
        yield d
        d += timedelta(days=1)

def upsert_reserva_conf_rango(
    fecha_inicio: date,
    fecha_fin: date,
    carpeta_prg: str | Path = "Programas/PRG",
    patron: str = "PRG{yy}{mm}{dd}.xlsx",
):
    """
    Recorre fechas [fecha_inicio..fecha_fin], busca PRG por fecha (con -1/-2) y sube a SQL.
    - carpeta_prg: carpeta donde están los PRG
    - patron: cómo se llama el archivo base (sin -1/-2). Por defecto PRGYYMMDD.xlsx
    """
    carpeta_prg = Path(carpeta_prg)

    ok = 0
    no_file = 0
    errors = 0

    for f in _daterange(fecha_inicio, fecha_fin):
        base = patron.format(yy=f.strftime("%y"), mm=f.strftime("%m"), dd=f.strftime("%d"))
        path = carpeta_prg / base
        try:
            upsert_reserva_conf(path)
            ok += 1
        except FileNotFoundError:
            print(f"[NO FILE] {f} -> {path.name}")
            no_file += 1
        except Exception as e:
            print(f"[ERROR] {f} -> {path.name}: {e}")
            errors += 1

    print(f"Resumen: OK={ok}, NO_FILE={no_file}, ERROR={errors}")


# -----------------------
# USO
# -----------------------
upsert_reserva_conf_rango(
     fecha_inicio=date(2026, 3, 3),
     fecha_fin=date(2026, 3, 16),
     carpeta_prg="Programas/PRG",
     patron="PRG{yy}{mm}{dd}.xlsx"
 )


OK: upsert 36456 filas en planificacion.reserva_conf (PRG260303.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260304.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260305.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260306.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260307.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260308.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260309.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260310.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260311.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260312.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260313.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260314.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260315.xlsx)
OK: upsert 36456 filas en planificacion.reserva_conf (PRG260316.xlsx)
Resumen: OK=14, NO_F

In [53]:
import mysql.connector

MYSQL = dict(host="127.0.0.1", port=3306, user="root", password="rootvale123")

def column_exists(cur, schema: str, table: str, column: str) -> bool:
    cur.execute("""
        SELECT COUNT(*)
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA=%s AND TABLE_NAME=%s AND COLUMN_NAME=%s
    """, (schema, table, column))
    return cur.fetchone()[0] > 0

def add_and_fill_subtipo():
    conn = mysql.connector.connect(**MYSQL)
    try:
        cur = conn.cursor()

        # 1) Crear columna Subtipo si no existe
        if not column_exists(cur, "planificacion", "reserva_conf", "Subtipo"):
            cur.execute("""
                ALTER TABLE planificacion.reserva_conf
                ADD COLUMN Subtipo VARCHAR(100) NULL
            """)
            conn.commit()

        # 2) Poblarla con JOIN (forzando collation para evitar el 1267)
        cur.execute("""
            UPDATE planificacion.reserva_conf rc
            JOIN (
              SELECT Central, MAX(Subtipo) AS Subtipo
              FROM costos_variables.prg_energia_horaria
              WHERE Central IS NOT NULL AND Central <> ''
              GROUP BY Central
            ) prg
              ON rc.Central COLLATE utf8mb4_0900_ai_ci
               = prg.Central COLLATE utf8mb4_0900_ai_ci
            SET rc.Subtipo = prg.Subtipo
        """)
        conn.commit()

        # (opcional) ver cuántas quedaron sin subtipo
        cur.execute("""
            SELECT COUNT(*)
            FROM planificacion.reserva_conf
            WHERE Subtipo IS NULL OR Subtipo=''
        """)
        print("Filas sin Subtipo:", cur.fetchone()[0])

    finally:
        conn.close()

add_and_fill_subtipo()


Filas sin Subtipo: 8280


In [54]:
import mysql.connector

MYSQL = dict(host="127.0.0.1", port=3306, user="root", password="rootvale123")

CENTRALES_CARBON = [
    "ANGAMOS-ANG1_CAR + BESS",
    "ANGAMOS-ANG2_CAR + BESS",
    "COCHRANE-CCH1_CAR + BESS",
    "COCHRANE-CCH2_CAR + BESS",
]

def set_subtipo_carbon_en_reserva_conf():
    conn = mysql.connector.connect(**MYSQL)
    try:
        cur = conn.cursor()
        placeholders = ",".join(["%s"] * len(CENTRALES_CARBON))

        # Forzamos misma collation en ambos lados para evitar 1267 (ajusta si tu DB usa otra)
        sql = f"""
            UPDATE planificacion.reserva_conf rc
            SET rc.Subtipo = 'Carbon'
            WHERE rc.Central COLLATE utf8mb4_0900_ai_ci IN (
                SELECT x.Central COLLATE utf8mb4_0900_ai_ci
                FROM (
                    SELECT %s AS Central
                    UNION ALL SELECT %s
                    UNION ALL SELECT %s
                    UNION ALL SELECT %s
                ) x
            );
        """

        cur.execute(sql, tuple(CENTRALES_CARBON))
        conn.commit()

        cur.execute(f"""
            SELECT Central, COUNT(*) AS n
            FROM planificacion.reserva_conf
            WHERE Central IN ({placeholders})
            GROUP BY Central
            ORDER BY Central;
        """, tuple(CENTRALES_CARBON))
        print("Filas afectadas por central:")
        for central, n in cur.fetchall():
            print(f"- {central}: {n}")

    finally:
        conn.close()

set_subtipo_carbon_en_reserva_conf()


Filas afectadas por central:
- ANGAMOS-ANG1_CAR + BESS: 10464
- ANGAMOS-ANG2_CAR + BESS: 10464
- COCHRANE-CCH1_CAR + BESS: 10464
- COCHRANE-CCH2_CAR + BESS: 10464


In [55]:
import re
import pandas as pd
import mysql.connector

# =========================
# CONFIG
# =========================
MYSQL = dict(host="127.0.0.1", port=3306, user="root", password="rootvale123")

T_SRC = "planificacion.reserva_conf"     # origen (normalizada)
DB_OUT = "planificacion"
T_OUT = "reservas_tec"                  # destino: planificacion.reservas_tec

# Mapeo TipoReserva -> prefijo
TIPO2PREF = {
    "Primaria": "CPF",
    "Secundaria": "CSF",
    "Terciaria": "CTF",
}

KEY_COLS = ["Fecha", "hora"]

# =========================
# HELPERS
# =========================
def q_ident(s: str) -> str:
    return "`" + s.replace("`", "") + "`"

def col_safe(s: str) -> str:
    """
    Mantiene legible, pero limpia caracteres problemáticos para nombres de columna.
    Ej: 'Gas Natural' -> 'Gas_Natural', 'Diésel' -> 'Diesel'
    """
    s = (s or "").strip().replace("\u00A0", " ")
    # quitar acentos comunes
    s = s.translate(str.maketrans("áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnN"))
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^0-9A-Za-z_+\-]", "", s)
    if not s:
        s = "SIN_SUBTIPO"
    if s[0].isdigit():
        s = "_" + s
    return s

def build_reservas_tec_df(conf: pd.DataFrame) -> pd.DataFrame:
    """
    Devuelve DF ancho con columnas:
      Fecha, hora,
      CPF+_<Subtipo>, CPF-_<Subtipo>, CSF+_<Subtipo>, CSF-_<Subtipo>, CTF+_<Subtipo>, CTF-_<Subtipo>
    donde '+' usa SUBIDA y '-' usa BAJADA.
    """
    df = conf.copy()

    # normalización mínima
    df["TipoReserva"] = df["TipoReserva"].astype(str).str.strip()
    df["Subtipo"] = df["Subtipo"].astype(str).str.strip().replace({"": "SIN_SUBTIPO"})
    df["Fecha"] = pd.to_datetime(df["Fecha"]).dt.date
    df["hora"] = df["hora"].astype(int)
    df["SUBIDA"] = pd.to_numeric(df["SUBIDA"], errors="coerce").fillna(0.0)
    df["BAJADA"] = pd.to_numeric(df["BAJADA"], errors="coerce").fillna(0.0)

    # filtrar solo tipos que mapean
    df = df[df["TipoReserva"].isin(TIPO2PREF.keys())].copy()
    if df.empty:
        return pd.DataFrame(columns=[*KEY_COLS])

    # agregar prefijo CPF/CSF/CTF
    df["PREF"] = df["TipoReserva"].map(TIPO2PREF)

    # agrupar (sum) por Fecha,hora,PREF,Subtipo
    g = (
        df.groupby(["Fecha", "hora", "PREF", "Subtipo"], as_index=False)[["SUBIDA", "BAJADA"]]
          .sum()
    )

    # pivot SUBIDA -> columnas "PREF+_<Subtipo>"
    sub = g.pivot_table(
        index=["Fecha", "hora"],
        columns=["PREF", "Subtipo"],
        values="SUBIDA",
        aggfunc="sum",
        fill_value=0.0,
    )

    # pivot BAJADA -> columnas "PREF-_<Subtipo>"
    baj = g.pivot_table(
        index=["Fecha", "hora"],
        columns=["PREF", "Subtipo"],
        values="BAJADA",
        aggfunc="sum",
        fill_value=0.0,
    )

    # a columnas planas y "seguras"
    def flatten_cols(multi_cols, sign: str) -> list[str]:
        out = []
        for pref, subt in multi_cols:
            out.append(f"{pref}{sign}_{col_safe(subt)}")
        return out

    sub.columns = flatten_cols(sub.columns, "+")
    baj.columns = flatten_cols(baj.columns, "-")

    wide = pd.concat([sub, baj], axis=1).reset_index()

    # ordenar columnas: Fecha,hora, luego por prefijo y signo
    def sort_key(c: str):
        if c in ("Fecha", "hora"):
            return (0, c)
        # ejemplo: CPF+_Carbon
        m = re.match(r"^(CPF|CSF|CTF)([+\-])_", c)
        if not m:
            return (9, c)
        pref, sign = m.group(1), m.group(2)
        pref_order = {"CPF": 1, "CSF": 2, "CTF": 3}[pref]
        sign_order = {"+": 1, "-": 2}[sign]
        return (pref_order, sign_order, c)

    cols_sorted = sorted(wide.columns, key=sort_key)
    wide = wide[cols_sorted]

    return wide

def ensure_table(conn, df: pd.DataFrame):
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE IF NOT EXISTS `{DB_OUT}`;")

    value_cols = [c for c in df.columns if c not in KEY_COLS]
    cols_sql = ",\n  ".join([f"{q_ident(c)} DECIMAL(18,6) NULL" for c in value_cols])

    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS `{DB_OUT}`.`{T_OUT}` (
          Fecha DATE NOT NULL,
          hora  TINYINT NOT NULL,
          {cols_sql},
          PRIMARY KEY (Fecha, hora)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)

    # asegurar columnas si aparecen nuevas
    for c in value_cols:
        try:
            cur.execute(f"ALTER TABLE `{DB_OUT}`.`{T_OUT}` ADD COLUMN {q_ident(c)} DECIMAL(18,6) NULL;")
        except mysql.connector.Error:
            pass

    conn.commit()

def upsert_df(conn, df: pd.DataFrame, chunk_size: int = 5000):
    if df.empty:
        print("DF vacío: nada que subir.")
        return

    key_cols = KEY_COLS
    value_cols = [c for c in df.columns if c not in key_cols]

    cols_insert = [*key_cols, *value_cols]
    placeholders = ", ".join(["%s"] * len(cols_insert))
    updates = ", ".join([f"{q_ident(c)}=VALUES({q_ident(c)})" for c in value_cols])

    sql = f"""
        INSERT INTO `{DB_OUT}`.`{T_OUT}` ({", ".join(q_ident(c) for c in cols_insert)})
        VALUES ({placeholders})
        ON DUPLICATE KEY UPDATE
          {updates};
    """

    cur = conn.cursor()

    rows = df.itertuples(index=False, name=None)
    batch = []
    n = 0
    for r in rows:
        # r ya viene en el orden df.columns
        # convertimos a tuple con float/None para valores
        r = list(r)
        # Fecha,hora están primero por construcción
        for j, c in enumerate(df.columns):
            if c in key_cols:
                continue
            v = r[j]
            r[j] = None if pd.isna(v) else float(v)
        batch.append(tuple(r))
        if len(batch) >= chunk_size:
            cur.executemany(sql, batch)
            conn.commit()
            n += len(batch)
            batch = []

    if batch:
        cur.executemany(sql, batch)
        conn.commit()
        n += len(batch)

    print(f"OK: upsert {n} filas en {DB_OUT}.{T_OUT}")

# =========================
# PIPELINE
# =========================
def run_build_and_upload():
    conn = mysql.connector.connect(**MYSQL)
    try:
        # leer solo columnas necesarias
        conf = pd.read_sql(
            f"SELECT Fecha, hora, TipoReserva, SUBIDA, BAJADA, Subtipo FROM {T_SRC}",
            conn
        )

        reservas_tec_df = build_reservas_tec_df(conf)

        ensure_table(conn, reservas_tec_df)
        upsert_df(conn, reservas_tec_df)

        print("Columnas creadas:", len(reservas_tec_df.columns))
        print("Primeras 15:", list(reservas_tec_df.columns[:15]))

    finally:
        conn.close()

run_build_and_upload()


/var/folders/b1/j2t3b3w12dl60tdztfsl3b0w0000gn/T/ipykernel_31460/3641770432.py:204: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  conf = pd.read_sql(


OK: upsert 10464 filas en planificacion.reservas_tec
Columnas creadas: 70
Primeras 15: ['Fecha', 'hora', 'CPF+_Carbon', 'CPF+_Diesel', 'CPF+_Embalse', 'CPF+_Eolica', 'CPF+_GNA', 'CPF+_GNL', 'CPF+_GNL_INF', 'CPF+_None', 'CPF+_Retiro', 'CPF+_Solar', 'CPF-_Carbon', 'CPF-_Diesel', 'CPF-_Embalse']
